## השוואה בין שני מודלים

יש לנו שני מודלים אפשריים לאותם נתונים (טווח מול $\sin(2\theta)$):

1. **מודל כללי**: $R = m \cdot \sin(2\theta) + b$ (2 פרמטרים חופשיים).
2. **מודל דרך הראשית**: $R = m \cdot \sin(2\theta)$ (פרמטר חופשי אחד -- `b` מוכרח להיות 0).

למודל השני יש **הצדקה פיזיקלית**: כש-$\theta=0$, $\sin(2\theta)=0$, וגם הטווח האמיתי חייב להיות 0 (זריקה אופקית בגובה הקרקע לא זזה). השאלה: האם שווה "לשלם" בפרמטר נוסף (`b`) עבור השיפור שהוא נותן?

In [ ]:
import numpy as np
import pandas as pd

g = 9.8
df = pd.read_csv("lab_measurements.csv")
df_clean = df.dropna()
angles = sorted(df_clean["angle_deg"].unique())

x = np.array([np.sin(2*np.radians(a)) for a in angles])
y = np.array([df_clean[df_clean["angle_deg"] == a]["range_measured"].mean() for a in angles])
sigma_y = np.array([
    df_clean[df_clean["angle_deg"] == a]["range_measured"].std(ddof=1) / np.sqrt(len(df_clean[df_clean["angle_deg"] == a]))
    for a in angles
])

# מודל כללי (2 פרמטרים)
x_bar, y_bar = x.mean(), y.mean()
m_general = np.sum((x - x_bar)*(y - y_bar)) / np.sum((x - x_bar)**2)
b_general = y_bar - m_general*x_bar
pred_general = m_general*x + b_general

# מודל דרך הראשית (פרמטר אחד)
m_origin = np.sum(x*y) / np.sum(x*x)
pred_origin = m_origin*x

print(f"כללי:        m={m_general:.3f}, b={b_general:.3f}")
print(f"דרך הראשית:  m={m_origin:.3f}")

### השוואה: R² מול חי-בריבוע מצומצם

In [ ]:
def r_squared(y, pred):
    ss_res = np.sum((y - pred)**2)
    ss_tot = np.sum((y - y.mean())**2)
    return 1 - ss_res/ss_tot

def reduced_chi2(y, pred, sigma_y, n_params):
    resid = y - pred
    chi2 = np.sum((resid/sigma_y)**2)
    dof = len(y) - n_params
    return chi2 / dof

r2_general = r_squared(y, pred_general)
r2_origin = r_squared(y, pred_origin)
rchi2_general = reduced_chi2(y, pred_general, sigma_y, n_params=2)
rchi2_origin = reduced_chi2(y, pred_origin, sigma_y, n_params=1)

print(f"מודל כללי:       R^2={r2_general:.4f}   chi^2_nu={rchi2_general:.3f}")
print(f"מודל דרך הראשית: R^2={r2_origin:.4f}   chi^2_nu={rchi2_origin:.3f}")

ה-$R^2$ כמעט זהה בין שני המודלים -- הפרמטר הנוסף (`b`) כמעט ולא "קנה" שיפור. אבל $\chi^2_\nu$ מספר סיפור אחר: המודל הפשוט יותר (דרך הראשית) נותן $\chi^2_\nu$ **קרוב יותר ל-1**, כי הוא לא "בזבז" דרגת חופש על פרמטר שכמעט ולא תרם. זו בדיוק הסיבה ש-$\chi^2_\nu$ (שמעניש לפי מספר הפרמטרים, דרך `dof`) עדיף על $R^2$ **לבדו** כשמשווים מודלים במספר פרמטרים שונה.

### באג נפוץ: לבחור מודל לפי R² בלבד

$R^2$ **לעולם לא יורד** כשמוסיפים פרמטר חופשי נוסף למודל (במקרה הגרוע ביותר הוא נשאר זהה) -- גם אם הפרמטר לא מוסיף שום מידע פיזיקלי אמיתי. לכן "לבחור את המודל עם ה-$R^2$ הגבוה ביותר" מטה את הבחירה באופן שיטתי לכיוון מודלים מורכבים יותר, גם כשהם לא באמת טובים יותר.

In [ ]:
print(f"R^2 כללי ({2} פרמטרים):        {r2_general:.6f}")
print(f"R^2 דרך הראשית ({1} פרמטר):    {r2_origin:.6f}")
print("R^2 הכללי גבוה יותר (או שווה) כמעט תמיד - זו לא הוכחה שהוא המודל הנכון יותר.")

### נסו בעצמכם

חשבו את שגיאת השיפוע (`SE_m`, כמו בסעיף 11.7) עבור שני המודלים, והשוו: האם הפרמטר `b` הנוסף במודל הכללי משנה משמעותית את אי-הוודאות בשיפוע עצמו?

In [ ]:
# ...

`````{admonition} פתרון
:class: dropdown, tip
```python
resid_general = y - pred_general
s_y_general = np.sqrt(np.sum(resid_general**2) / (len(x) - 2))
SE_m_general = s_y_general / np.sqrt(np.sum((x - x_bar)**2))

resid_origin = y - pred_origin
s_y_origin = np.sqrt(np.sum(resid_origin**2) / (len(x) - 1))
SE_m_origin = s_y_origin / np.sqrt(np.sum(x**2))

print(f"SE_m (כללי):       {SE_m_general:.3f}")
print(f"SE_m (דרך הראשית): {SE_m_origin:.3f}")
```
`````

### בדקו את עצמכם

In [ ]:
from jupyterquiz import display_quiz

questions = [
    {
        "question": "למה R^2 אינו כלי טוב, לבדו, להשוואה בין מודלים עם מספר פרמטרים שונה?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "כי הוא לעולם לא יורד כשמוסיפים פרמטרים, ולכן תמיד 'מעדיף' את המודל המורכב יותר, בלי קשר לתועלת האמיתית", "correct": True, "feedback": "נכון."},
            {"answer": "כי R^2 לא ניתן לחישוב עבור מודל דרך הראשית", "correct": False, "feedback": "ראינו שכן ניתן לחשב אותו."},
            {"answer": "אין שום בעיה - R^2 הוא הכלי הטוב ביותר להשוואת מודלים בכל מקרה", "correct": False, "feedback": "לא נכון - chi^2_nu מתחשב במספר הפרמטרים, R^2 לא."}
        ]
    }
]
display_quiz(questions)

### תרגול עצמי

חשבו את ה-**הפרש** ב-$\chi^2$ (לא המצומצם -- ה-$\chi^2$ הגולמי) בין שני המודלים, וגם את ההפרש ב-dof (מספר דרגות החופש). זהו בסיס לרעיון סטטיסטי מתקדם יותר (מבחן $\chi^2$ להשוואת מודלים מקוננים) שלא נעסוק בו כאן לעומק, אבל שווה להכיר את הכיוון.

In [ ]:
# ...

`````{admonition} פתרון
:class: dropdown, tip
```python
chi2_general = np.sum(((y - pred_general)/sigma_y)**2)
chi2_origin = np.sum(((y - pred_origin)/sigma_y)**2)
delta_chi2 = chi2_origin - chi2_general
delta_dof = 1   # מודל אחד עם פרמטר אחד פחות

print(f"chi^2 כללי: {chi2_general:.3f},  chi^2 דרך הראשית: {chi2_origin:.3f}")
print(f"הפרש: {delta_chi2:.3f}  (דרגת חופש הפרש: {delta_dof})")
```
`````